# Raqam darvozasi (self-contained)

Kataklarni yuqoridan pastga ishga tushir. Kernel repo ildizida (`TurnDetecting/`) bo'lsin — 3-katak `uz_dataset/` ni topishi uchun.

In [1]:
# -*- coding: utf-8 -*-
"""
O'zbekcha raqam diktovkasi uchun deterministik "slot-filling" darvoza.

Muammo: EOU (turn-detector) modeli GRAMMATIK tugaganlikni baholaydi, raqam
diktovkasi esa SANASH masalasi ("yetti yuz o'n yigirma ikki" = 5 raqam, passport
7 raqam bo'lishi kerak -> hali tugamagan). Model buni ishonchli qila olmaydi.

Yechim: call-center oqimida qaysi maydon yig'ilayotgani ma'lum (assistant savolini
o'zing boshqarasan). Foydalanuvchi aytgan raqam so'zlarini raqamga aylantirib,
sonini deterministik tekshirasan. Tugaganlikni model emas, shu darvoza hal qiladi.

Asosiy funksiyalar:
    uz_words_to_digits("yetti yuz o'n yigirma ikki")  -> "71022"
    should_endpoint(user_text, collecting="passport_num", eou_prob=...) -> bool

STT ning uch formatini ham qo'llaydi:
    bo'lak-so'z:    "yetti yuz o'n yigirma ikki yetmish uch" -> "7102273"
    raqamma-raqam:  "yetti bir nol ikki ikki yetti uch"      -> "7102273"
    yaxlit raqam:   "7102273"                                 -> "7102273"
    aralash/prefiks:"raqamim 91 695 34 56"                    -> "916953456"
    harflar (passport seriyasi) e'tiborsiz qoldiriladi:
                    "ab 1234567" / "a be bir ikki uch ..."    -> "1234567"

LiveKit oqimiga ulash (soddalashtirilgan):

    collecting = None       # hozir qaysi raqam maydoni yig'ilyapti (None = erkin nutq)

    def on_agent_turn(agent_text):          # agent savol berganda
        nonlocal collecting
        collecting = detect_field(agent_text)   # yoki o'z oqim holatingdan ber

    def on_user_eou(user_text, eou_prob):   # STT to'liq transkript + EOU ehtimoli
        if should_endpoint(user_text, collecting, eou_prob):
            collecting = None               # maydon to'ldi -> holatni tozala
            return respond()
        return keep_listening()             # sukut taymerini uzaytirib kutamiz

Muhim maslahatlar:
  - Raqam maydonlarida sukut taymerini uzaytir (masalan 1.5-2s) — darvoza "kutamiz"
    deganda ham odam raqamni tugatib ulgursin.
  - Kritik maydonlar (passport/karta/jshshir) uchun read-back tasdiq qo'sh:
    "raqamingiz ... to'g'rimi?" — noto'g'ri uzunlik shu yerda ushlanadi.
  - So'z bilan aytilgan raqamlarni ishonchli sanash ID uslublarida ishlaydi
    (raqamma-raqam, 2-xonali juft, yaxlit). Noaniq "yumaloq" o'qishlar
    ("uch yuz sakson olti") tabiatan noaniq — bunda read-back kafolat beradi.
"""

import re
import unicodedata

# ---------------------------------------------------------------------------
# Normalizatsiya (dataset generatoridagi bilan bir xil) — apostrof variantlari
# ASCII ' ga keltiriladi, kichik harf, punktuatsiya olib tashlanadi.
# ---------------------------------------------------------------------------
_APOS = "ʻʼ‘’`´"


def _normalize(text: str) -> str:
    if not text:
        return ""
    for ch in _APOS:
        text = text.replace(ch, "'")
    text = unicodedata.normalize("NFKC", text.lower())
    text = "".join(ch for ch in text
                   if not (unicodedata.category(ch).startswith("P") and ch not in ["'", "-"]))
    return re.sub(r"\s+", " ", text).strip()


# ---------------------------------------------------------------------------
# Raqam-so'z lug'ati. Apostrofsiz variantlar ham (STT apostrofni tushirishi mumkin).
# ---------------------------------------------------------------------------
# Nol (0) ning yozilish/STT variantlari — alohida qaymiz (leading zero uchun)
ZERO_WORDS = {"nol", "no'l", "nul", "noil", "nöl"}
UNITS = {
    "nol": 0, "no'l": 0, "nul": 0, "bir": 1, "ikki": 2, "uch": 3,
    "to'rt": 4, "tort": 4, "besh": 5, "olti": 6, "yetti": 7,
    "sakkiz": 8, "to'qqiz": 9, "toqqiz": 9,
}
TENS = {
    "o'n": 10, "on": 10, "yigirma": 20, "o'ttiz": 30, "ottiz": 30, "qirq": 40,
    "ellik": 50, "oltmish": 60, "yetmish": 70, "sakson": 80,
    "to'qson": 90, "toqson": 90,
}
HUND = {"yuz": 100}
# Diqqat: bu darvoza ID RAQAM ketma-ketligi uchun (passport/telefon/jshshir/karta).
# "ming" (1000) ID o'qishda uchramaydi — u faqat yil/summa uchun; shuning uchun
# lug'atga kiritilmagan (uchrasa chegara so'zi sifatida qaraladi). Yil tugaganligi
# raqam soni bilan emas, kontekst bilan (model) hal qilinadi.

# "lik" qo'shimchasi (operator kodi belgisi): user "to'qson birlik" (91-lik),
# "to'qsonlik" (90-lik) deydi — shu kompaniyaga tegishli degani. Bu raqam GURUHINI
# yakunlaydi: "to'qsonlik olti yuz..." -> 90 | 6.. (aks holda "to'qson olti"=96
# bo'lib qo'shilib ketardi). Qo'shimchani ajratamiz va guruhni majburan yopamiz.
_LIK_SUFFIXES = ("talik", "taligi", "ligi", "lig'i", "lig", "lik")


def _strip_lik(tok: str):
    """Raqam so'zidagi 'lik' qo'shimchasini ajratadi.
    Qaytaradi: (o'zak, True) agar qo'shimcha bo'lsa, aks holda (tok, False).
    'ellik' (50) kabi to'g'ridan-to'g'ri raqamlarga tegmaydi."""
    if tok in UNITS or tok in TENS or tok in HUND:
        return tok, False          # 'ellik', 'birlik'? -> avval to'g'ridan tekshir
    for suf in _LIK_SUFFIXES:
        stem = tok[:-len(suf)]
        if tok.endswith(suf) and (stem in UNITS or stem in TENS or stem in HUND):
            return stem, True
    return tok, False


def uz_words_to_digits(text: str) -> str:
    """O'zbekcha aytilgan raqamni raqam SATRiga aylantiradi (leading zero saqlanadi).
    Qaytadagi satr uzunligi = aytilgan raqamlar soni. Kirish — STT so'z chiqishi
    ("to'qson birlik olti yuz to'qson besh ..."), literal raqam kutilmaydi (lekin
    kelib qolsa ham qo'llanadi)."""
    tokens = _normalize(text).split()
    out = []              # yig'ilgan raqam bo'laklari
    cur = 0               # joriy guruh qiymati
    seen_tens = False
    seen_units = False
    active = False

    def flush():
        nonlocal cur, seen_tens, seen_units, active
        if active:
            out.append(str(cur))
        cur, seen_tens, seen_units, active = 0, False, False, False

    for raw in tokens:
        if raw.isdigit():                 # STT'da kutilmaydi; xavfsizlik uchun qoldirilgan
            flush()
            out.append(raw)
            continue
        tok, force_flush = _strip_lik(raw)
        if tok in ZERO_WORDS:             # 0 doim yangi guruh boshlaydi (leading zero)
            flush()
            out.append("0")
        elif tok in UNITS:
            if seen_units:                # ketma-ket birlik -> yangi guruh (raqamma-raqam)
                flush()
            cur += UNITS[tok]
            seen_units, active = True, True
        elif tok in TENS:
            if seen_tens or seen_units:   # o'nlik allaqachon bor edi -> yangi guruh
                flush()
            cur += TENS[tok]
            seen_tens, active = True, True
        elif tok in HUND:                 # "olti yuz" = 6*100
            if not active:
                cur = 1
            cur *= 100
            seen_tens = seen_units = False
            active = True
        else:                             # harf / to'ldiruvchi so'z -> guruh chegarasi
            flush()
            continue
        if force_flush:                   # 'lik' -> operator kodi guruhi yopildi
            flush()
    flush()
    return "".join(out)


In [2]:
# ---------------------------------------------------------------------------
# Darvoza: aniq uzunlikdagi raqam maydonlari
# ---------------------------------------------------------------------------
EXPECTED = {
    "passport_num": 7,    # passport/ID raqami (seriya harflarisiz)
    "phone": 9,           # operator kodi (2) + 7  — 998 prefiksisiz
    "jshshir": 14,        # PINFL
    "card": 16,           # Uzcard/Humo
    "card_last4": 4,      # oxirgi 4 raqam
}
# Eslatma: tug'ilgan yil/sana bu yerda YO'Q — u aniq-uzunlik masalasi emas
# (kun-oy-yil maydonlari, kontekstga bog'liq). Uni EOU model + savol turi hal qiladi.

# Assistant savolidan maydonni taxmin qilish uchun kalit so'zlar (ixtiyoriy yordamchi —
# amalda holatni o'z oqim mantiqing beradi, bu faqat qulaylik uchun).
_FIELD_HINTS = [
    ("card_last4", ["oxirgi to'rt", "oxirgi 4"]),
    ("jshshir", ["jshshir", "pinfl", "jshshshir"]),
    ("card", ["karta raqam", "o'n olti xonali", "16 xonali"]),
    ("passport_num", ["passport raqam", "hujjat raqam", "raqamlarini", "yetti xonali"]),
    ("phone", ["telefon", "raqamga sms", "aloqa uchun raqam", "bog'lanish uchun raqam"]),
    ("birth_year", ["tug'ilgan yil", "necha yilda", "qaysi yili"]),
]


def detect_field(assistant_text: str):
    """Assistant savolidan qaysi raqam maydoni so'ralganini taxmin qiladi.
    Topilmasa None (ya'ni bu raqam-yig'ish holati emas)."""
    t = _normalize(assistant_text)
    for field, keys in _FIELD_HINTS:
        if any(k in t for k in keys):
            return field
    return None


def should_endpoint(user_text: str,
                    collecting=None,
                    eou_prob: float = 0.0,
                    eou_threshold: float = 0.05) -> bool:
    """Foydalanuvchi gapini tugatdimi (agent javob bersa bo'ladimi)?

    collecting: hozir yig'ilayotgan raqam maydoni nomi (EXPECTED kaliti) yoki None.
      - Raqam maydoni bo'lsa: qaror DETERMINISTIK — raqamlar soni yetganda True.
      - None (erkin nutq) bo'lsa: EOU modelga ishonamiz (eou_prob >= threshold).
    """
    if collecting in EXPECTED:
        n = len(uz_words_to_digits(user_text))
        need = EXPECTED[collecting]
        if n < need:
            return False          # chala raqam — EOU nima desa ham KUTAMIZ
        return True               # >= kerakli son — to'ldi (ortiqcha bo'lsa ham to'xtatamiz)
    return eou_prob >= eou_threshold


# ---------------------------------------------------------------------------
# O'z-o'zini tekshirish: generatorning o'qish funksiyalariga qarshi property-test
# ---------------------------------------------------------------------------


In [7]:
import random
import sys
from pathlib import Path

try:
    _here = Path(__file__).resolve().parent
except NameError:
    _here = Path.cwd()   # notebook: kernel repo ildizida
sys.path.insert(0, str(_here / "uz_dataset"))
from generate_uz_eou_dataset import read_chunks, group_words, digit_words

rng = random.Random(0)

def pairwise_words(d):    # jshshir uslubi: juftlab (juft uzunlik uchun)
    return " ".join(group_words(d[i:i + 2]) for i in range(0, len(d), 2))

# 1) Property-test: STT SO'Z chiqishi 100% mos kelishi SHART.
#    (literal raqam kutilmaydi — TTS/STT doim so'z beradi)
#    - digit: raqamma-raqam ("bir ikki uch")
#    - pair:  juftlab (faqat juft uzunlik; ID maydonlari 14/16 juft)
#    - chunk: read_chunks (noaniqlikdan xoli qilingan)
reliable_fail = 0
chunk_ok = chunk_tot = 0
for _ in range(20000):
    length = rng.choice([7, 9, 14, 16, 4])
    d = "".join(str(rng.randrange(10)) for _ in range(length))
    reliable = {"digit": digit_words(d)}
    if length % 2 == 0:
        reliable["pair"] = pairwise_words(d)
    for name, text in reliable.items():
        if uz_words_to_digits(text) != d:
            reliable_fail += 1
            if reliable_fail <= 10:
                print(f"FAIL [{name}] {d!r} -> {uz_words_to_digits(text)!r}  ({text!r})")
    chunk_tot += 1
    chunk_ok += (uz_words_to_digits(" ".join(read_chunks(d, rng))) == d)
print(f"\nso'z uslublari (digit/pair): "
      f"{'OK — 100%' if reliable_fail == 0 else str(reliable_fail) + ' FAIL'}")
print(f"chunk (read_chunks) aniqligi: {chunk_ok/chunk_tot*100:.1f}%  "
      f"({chunk_ok}/{chunk_tot})")

# 2) Real STT holatlari (hammasi SO'Z; 'lik' operator kodi belgisi bilan)
cases = [
    # foydalanuvchi misollari — passport (7 raqam)
    ("yetti yuz o'n yigirma ikki", "71022"),                       # chala
    ("yetti yuz o'n yigirma ikki yetmish uch", "7102273"),         # to'liq
    ("a d yetti yuz o'n yigirma ikki yetmish uch", "7102273"),     # harflar bilan
    ("nol besh to'rt sakkiz ikki to'qqiz to'qqiz", "0548299"),     # nol bilan
    ("to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti", "916953456"),
    ("to'qsonlik olti yuz to'qson besh o'ttiz to'rt ellik olti", "906953456"),
    ("o'ttiz uchlik olti yuz to'qson besh o'ttiz to'rt ellik olti", "336953456"),
    ("yetmish yettilik ikki yuz o'n uch no'l to'rt ellik olti", "772130456"), 
    ("to'qson bir olti yuz to'qson besh o'ttiz to'rt ellik olti", "916953456"),
]
print("\nreal STT (so'z) holatlar:")
hand_fail = 0
for text, exp in cases:
    got = uz_words_to_digits(text)
    if exp is None:
        print(f"  ??  {got!r:>16}  <= {text!r}  (uzunlik={len(got)})")
        continue
    hand_fail += (got != exp)
    print(f"  {'OK ' if got == exp else 'XATO'} {got!r:>16}  <= {text!r}")

# 3) Darvoza namunasi
print("\ndarvoza (should_endpoint):")
demo = [
    ("passport_num", "yetti yuz o'n yigirma ikki", False),
    ("passport_num", "yetti yuz o'n yigirma ikki yetmish uch", True),
    ("phone", "to'qson birlik olti yuz to'qson besh o'ttiz to'rt", False),
    ("phone", "to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti", True),
    ("jshshir", "o'ttiz ikki ellik nol besh", False),
]
for field, text, exp in demo:
    got = should_endpoint(text, collecting=field, eou_prob=0.99)
    n = len(uz_words_to_digits(text))
    print(f"  {'OK ' if got == exp else 'XATO'} {field:12} {n:>2}/{EXPECTED[field]}  "
          f"endpoint={got}  <= {text!r}")



so'z uslublari (digit/pair): OK — 100%
chunk (read_chunks) aniqligi: 100.0%  (20000/20000)

real STT (so'z) holatlar:
  OK           '71022'  <= "yetti yuz o'n yigirma ikki"
  OK         '7102273'  <= "yetti yuz o'n yigirma ikki yetmish uch"
  OK         '7102273'  <= "a d yetti yuz o'n yigirma ikki yetmish uch"
  OK         '0548299'  <= "nol besh to'rt sakkiz ikki to'qqiz to'qqiz"
  OK       '916953456'  <= "to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti"
  OK       '906953456'  <= "to'qsonlik olti yuz to'qson besh o'ttiz to'rt ellik olti"
  OK       '336953456'  <= "o'ttiz uchlik olti yuz to'qson besh o'ttiz to'rt ellik olti"
  OK       '772130456'  <= "yetmish yettilik ikki yuz o'n uch no'l to'rt ellik olti"
  OK       '916953456'  <= "to'qson bir olti yuz to'qson besh o'ttiz to'rt ellik olti"

darvoza (should_endpoint):
  OK  passport_num  5/7  endpoint=False  <= "yetti yuz o'n yigirma ikki"
  OK  passport_num  7/7  endpoint=True  <= "yetti yuz o'n yigirma ikki yetm

In [11]:
print(repr(uz_words_to_digits("Yetti yuz o'n Yigirma ikki")))

'71022'


In [5]:
# --- Interaktiv demo ---
tests = [
    "yetti yuz o'n yigirma ikki",                       # passport chala (5)
    "yetti yuz o'n yigirma ikki yetmish uch",           # passport to'liq (7)
    "to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti",   # 91-lik
    "to'qsonlik olti yuz to'qson besh o'ttiz to'rt ellik olti",       # 90-lik
    "yetmish yettilik ikki yuz o'n uch no'l to'rt ellik olti",        # no'l varianti
]
for t in tests:
    d = uz_words_to_digits(t)
    print(f"{len(d):>2} raqam  {d:<13}  <=  {t}")

print()
for field, text in [
    ("passport_num", "yetti yuz o'n yigirma ikki"),
    ("passport_num", "yetti yuz o'n yigirma ikki yetmish uch"),
    ("phone",        "to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti"),
    ("jshshir",      "o'ttiz ikki ellik nol besh yetmish olti"),
]:
    n = len(uz_words_to_digits(text))
    ep = should_endpoint(text, collecting=field, eou_prob=0.99)
    print(f"{field:12} {n:>2}/{EXPECTED[field]}  ->  {'JAVOB BER' if ep else 'KUTAMIZ':10}  <=  {text}")


 5 raqam  71022          <=  yetti yuz o'n yigirma ikki
 7 raqam  7102273        <=  yetti yuz o'n yigirma ikki yetmish uch
 9 raqam  916953456      <=  to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti
 9 raqam  906953456      <=  to'qsonlik olti yuz to'qson besh o'ttiz to'rt ellik olti
 9 raqam  772130456      <=  yetmish yettilik ikki yuz o'n uch no'l to'rt ellik olti

passport_num  5/7  ->  KUTAMIZ     <=  yetti yuz o'n yigirma ikki
passport_num  7/7  ->  JAVOB BER   <=  yetti yuz o'n yigirma ikki yetmish uch
phone         9/9  ->  JAVOB BER   <=  to'qson birlik olti yuz to'qson besh o'ttiz to'rt ellik olti
jshshir       8/14  ->  KUTAMIZ     <=  o'ttiz ikki ellik nol besh yetmish olti
